In [1]:
from transformers import pipeline
import torch
import sys
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
import json
from tqdm import tqdm
import gc
import math

# Automated data generation

In [2]:
def create_synthetic_data_local(model_name, model, tokenizer, input_path, output_path, split=None):
    """
    Generate synthetic data using a locally loaded LLM
    
    Args:
        model_name: Name of the model from HuggingFace
        input_path: Path to input JSONL file
        output_path: Path to output JSONL file
        split: Optional tuple (start, end) to process only a subset of the input
    """
    
    
    # Read lines from the input file
    with open(input_path, "r", encoding="utf-8") as fin:
        all_lines = fin.readlines()
    
    # If `split` is set, slice accordingly
    if split is not None:
        all_lines = all_lines[split[0]:split[1]]
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Open output in append mode
    with open(output_path, "a", encoding="utf-8") as fout:
        for i, line in tqdm(enumerate(all_lines), total=len(all_lines), desc="Generating responses"):
            try:
                # Parse the JSON data
                data = json.loads(line)
                instruction = data["instruction"]
                
                # Prepare input for the model
                messages = [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": instruction}
                ]
                
                text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
                
                model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
                
                # Generate response with return_dict_in_generate=True to get probabilities
                generation_output = model.generate(
                    **model_inputs,
                    max_new_tokens=1024,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    return_dict_in_generate=True,
                    output_scores=True
                )
                
                generated_ids = generation_output.sequences
                scores = generation_output.scores
                
                # Extract only the generated part (not the input)
                input_length = model_inputs.input_ids.shape[1]
                generated_ids_only = [
                    output_ids[input_length:] for output_ids in generated_ids
                ]
                
                # Decode the generated tokens
                generated_text = tokenizer.batch_decode(generated_ids_only, skip_special_tokens=True)[0]
                
                # Get the token strings for each generated token
                generated_tokens = [tokenizer.decode(token_id.item()) for token_id in generated_ids_only[0]]
                
                # Compute probabilities, log‑probs, and sequence perplexity
                token_probabilities = []
                token_logprobs = []
                for step_idx, logits in enumerate(scores):
                    # Convert logits to log‑probabilities (temperature already applied)
                    log_probs = torch.nn.functional.log_softmax(logits[0], dim=-1)
                    token_id   = generated_ids_only[0][step_idx].item()
                    logp       = log_probs[token_id].item()
                    token_logprobs.append(logp)
                    token_probabilities.append(math.exp(logp))

                # Perplexity = exp( – average log‑prob )
                avg_neg_logp = -sum(token_logprobs) / len(token_logprobs)
                perplexity   = math.exp(avg_neg_logp)

                # Store everything in the data dict
                model_short = model_name.split("/")[-1]
                data[f"response_{model_short}"]           = generated_text
                data[f"tokens_{model_short}"]             = generated_tokens
                data[f"token_probabilities_{model_short}"]= token_probabilities
                data[f"token_logprobs_{model_short}"]     = token_logprobs
                data[f"perplexity_{model_short}"]         = perplexity
                
                # Write out the updated data immediately
                fout.write(json.dumps(data, ensure_ascii=False))
                fout.write("\n")
                
                # Every 10 lines, explicitly flush to disk
                if i % 10 == 0:
                    fout.flush()
                    
                # Print progress info
                if i % 10 == 0:
                    print(f"Processed {i}/{len(all_lines)} examples.")
                
            except Exception as e:
                print(f"Error on line {i}: {e}")
                break 
    
    print("Done generating synthetic data.")

In [3]:
# model_name = "meta-llama/Llama-3.1-8B-Instruct"
model_name = "Qwen/Qwen2.5-7B-Instruct"

torch.set_grad_enabled(False) # turn off for inference

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    use_safetensors=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [4]:
model.config.sliding_window = None

In [ ]:
# Configuration

dolly_version = 1
from_line = 754
till_line = 1125



# Set up paths
#input_path = f"../../Data/Finetuning/Dolly/dolly_train_{dolly_version}.jsonl"
input_path = f"../../../Data/Finetuning/Dolly/dolly_test.jsonl"



# Create a model-specific output path
model_short = model_name.split("/")[-1]
#output_path = f"../../Data/Finetuning/Augmented/Small/Llama_with_perplexity/dolly_train_{dolly_version}_{model_short}.jsonl"
output_path = f"../../../Data/Finetuning/Augmented/Small/Other/dolly_test_multi.jsonl"


print(f"Generating data with {model_name}")

create_synthetic_data_local(
    model_name=model_name,
    model=model,
    tokenizer=tokenizer,
    input_path=input_path,
    output_path=output_path,
    split=(from_line, till_line)
)


print(f"Completed generation with {model_name}")


# Clearing memory 

del model  
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()




Generating data with Qwen/Qwen2.5-7B-Instruct


Generating responses:   0%|          | 1/371 [00:03<24:02,  3.90s/it]

Processed 0/371 examples.


Generating responses:   3%|▎         | 11/371 [01:39<42:03,  7.01s/it] 

Processed 10/371 examples.


...

Generating responses: 100%|██████████| 371/371 [59:04<00:00,  9.55s/it]

Processed 370/371 examples.
Done generating synthetic data.
Completed generation with Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/652 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/29.9k [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

In [ ]:

# Configuration falcon

model_name = "tiiuae/Falcon3-10B-Instruct"

torch.set_grad_enabled(False) # turn off for inference

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
    use_safetensors=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

dolly_version = 1
from_line = 1125
till_line = 1500



# Set up paths
#input_path = f"../../Data/Finetuning/Dolly/dolly_train_{dolly_version}.jsonl"
input_path = f"../../../Data/Finetuning/Dolly/dolly_test.jsonl"



# Create a model-specific output path
model_short = model_name.split("/")[-1]
#output_path = f"../../Data/Finetuning/Augmented/Small/Llama_with_perplexity/dolly_train_{dolly_version}_{model_short}.jsonl"
output_path = f"../../../Data/Finetuning/Augmented/Small/Other/dolly_test_multi.jsonl"


print(f"Generating data with {model_name}")

create_synthetic_data_local(
    model_name=model_name,
    model=model,
    tokenizer=tokenizer,
    input_path=input_path,
    output_path=output_path,
    split=(from_line, till_line)
)


print(f"Completed generation with {model_name}")


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/365k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.78M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

...

Generating responses:   0%|          | 1/375 [00:01<09:49,  1.58s/it]Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Processed 0/375 examples.


Generating responses:   3%|▎         | 11/375 [01:24<1:11:56, 11.86s/it]Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Processed 10/375 examples.


Generating responses:   3%|▎         | 12/375 [01:32<1:04:13, 10.62s/it]Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


In [6]:
# Clearing memory 

del model  
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()